# Moosic — 05. Agglomerative (Hierarchical) Clustering

Bottom-up hierarchical clustering: every song starts in its own cluster, pairs merge step by step until everything's in one cluster. No density parameter (unlike DBSCAN), no forced pre-decided k the way K-Means needs one before fitting — but `n_clusters` (or a distance threshold) still has to be chosen to actually cut the hierarchy into usable groups.

**Central question for this notebook:** does this find a workable middle ground DBSCAN couldn't — many reasonably-sized, coherent clusters, not a fragmentation/mega-blob split — or does it confirm the same underlying "no natural playlist count exists in this data" conclusion a third way?

**Scaling note:** course example uses `StandardScaler`; keeping `MinMaxScaler` for consistency with `01`-`04`. Already confirmed with DBSCAN that scaler choice doesn't affect the qualitative comparison (silhouette is scale-relative).

**Structure note:** only **Step 6** writes to the shared `df['agg_cluster']` column that later steps depend on. Every linkage/parameter experiment (`5a`, `5b`, `5c`, etc.) gets its own uniquely-named variable — same convention adopted after the DBSCAN notebook mix-up.


## 0. Success Criteria (Carried Over)

Same two axes as DBSCAN — **coverage** (% of songs placed) and **coherence** (silhouette + spot-checks). Agglomerative, like K-Means, has no noise concept: every song lands in a cluster, always. So coverage is ~100% by construction here — this notebook is really a **coherence-only** comparison against K-Means and DBSCAN's non-noise clusters, not a coverage tradeoff like DBSCAN was.


## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.metrics import silhouette_score
from sklearn.manifold import TSNE
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn import set_config
import os

set_config(transform_output="pandas")
os.makedirs("../outputs", exist_ok=True)
RANDOM_STATE = 42

## 2. Load Data & Scale

In [ ]:
df = pd.read_csv("../data/5000_songs.csv")
df.columns = df.columns.str.strip()

features = ['danceability', 'energy', 'acousticness', 'tempo', 'valence',
            'speechiness', 'instrumentalness']

scaler = MinMaxScaler().set_output(transform="pandas")
scaled = scaler.fit_transform(df[features])

print(f"Shape: {df.shape}")

## 3. Truncated Dendrogram — a Rough Sanity Check, Not a Precise Count

5000 leaves would be completely unreadable in full — same problem as the many-category t-SNE plot and the earlier 125-cluster radar chart. `truncate_mode="level"` collapses the bottom of the tree so only the top few merge levels show, giving a rough sense of overall shape (does it look like a few big natural groups, or one long uneven chain?) without pretending to be precise.


In [ ]:
linked = linkage(scaled, method='ward')

plt.figure(figsize=(12, 6))
dendrogram(linked, truncate_mode="level", p=5, no_labels=True)
plt.title("Hierarchical Clustering Dendrogram (truncated — top 5 levels only)")
plt.xlabel("Songs (individual leaves not shown — too many to label)")
plt.ylabel("Ward Distance (variance increase on merge)")
plt.tight_layout()
plt.savefig("../outputs/05_dendrogram_truncated.png", dpi=150, bbox_inches="tight")
plt.show()

**What to look for:** per the course material, the biggest vertical gap with no horizontal line crossing it suggests a natural cut point. Note roughly where that gap sits (how many branches at that height) — this is a rough cross-check against Step 4's elbow/silhouette result, not a replacement for it.


## 4. Choosing k — Elbow + Silhouette

The course's own fallback for when a dendrogram is too complex to read precisely. Same tools already used for K-Means.


In [ ]:
k_values = range(2, 20)
silhouettes = []

for k in k_values:
    agg = AgglomerativeClustering(n_clusters=k, metric='euclidean', linkage='ward')
    labels = agg.fit_predict(scaled)
    silhouettes.append(silhouette_score(scaled, labels))

plt.figure(figsize=(9, 5))
plt.plot(k_values, silhouettes, 'go-')
plt.xlabel('k'); plt.ylabel('Silhouette Score')
plt.title('Agglomerative (Ward) — Silhouette by k')
plt.grid(True)
plt.tight_layout()
plt.savefig("../outputs/05_silhouette_ward.png", dpi=150, bbox_inches="tight")
plt.show()

pd.DataFrame({"k": list(k_values), "silhouette": [round(s, 3) for s in silhouettes]})

## 5. Linkage Criteria — Experiments

Ward (Step 4, above) is the course's emphasized default and the main run this notebook builds on. The three alternatives below are run as independent, uniquely-named experiments — none of them touch `scaled` or write to `df`, so they can be run in any order without side effects.

**Worth watching specifically:** single linkage is known for producing long, chain-like clusters — conceptually close to the "chaining" failure mode DBSCAN showed under very low `min_samples`. If it shows the same failure here, that's evidence the problem is about loosely-connected data in general, not something specific to density-based methods.


### 5a. Complete linkage

In [ ]:
silhouettes_complete = []
for k in k_values:
    agg_complete = AgglomerativeClustering(n_clusters=k, metric='euclidean', linkage='complete')
    labels_complete = agg_complete.fit_predict(scaled)
    silhouettes_complete.append(silhouette_score(scaled, labels_complete))

pd.DataFrame({"k": list(k_values), "silhouette_complete": [round(s, 3) for s in silhouettes_complete]})

### 5b. Average linkage

In [ ]:
silhouettes_average = []
for k in k_values:
    agg_average = AgglomerativeClustering(n_clusters=k, metric='euclidean', linkage='average')
    labels_average = agg_average.fit_predict(scaled)
    silhouettes_average.append(silhouette_score(scaled, labels_average))

pd.DataFrame({"k": list(k_values), "silhouette_average": [round(s, 3) for s in silhouettes_average]})

### 5c. Single linkage — watch for chaining

**Prediction to test:** expect either a trivially high silhouette caused by one dominant cluster absorbing almost everything (same shape as DBSCAN's misleading "recovery" near collapse), or visibly unstable/low scores from chain-like merging. Check cluster *sizes*, not just the silhouette number, same lesson as DBSCAN's eps=0.20 case.


In [ ]:
silhouettes_single = []
sizes_single = []
for k in k_values:
    agg_single = AgglomerativeClustering(n_clusters=k, metric='euclidean', linkage='single')
    labels_single = agg_single.fit_predict(scaled)
    silhouettes_single.append(silhouette_score(scaled, labels_single))
    sizes_single.append(pd.Series(labels_single).value_counts().sort_values(ascending=False).tolist())

for k, sil, sizes in zip(k_values, silhouettes_single, sizes_single):
    print(f"k={k}: silhouette={sil:.3f}, top sizes={sizes[:5]}{'...' if len(sizes) > 5 else ''}")

## 6. Chosen Configuration — the Canonical Run

In [ ]:
# SET based on Step 4's silhouette peak (Ward linkage, the main run this notebook builds on)
BEST_K = 8  # placeholder — replace with the actual peak k from Step 4's table
BEST_LINKAGE = 'ward'

agg_clustering = AgglomerativeClustering(n_clusters=BEST_K, metric='euclidean', linkage=BEST_LINKAGE)
df['agg_cluster'] = agg_clustering.fit_predict(scaled)

agg_silhouette = silhouette_score(scaled, df['agg_cluster'])
print(f"k={BEST_K}, linkage={BEST_LINKAGE}")
print(f"Silhouette: {agg_silhouette:.3f}")
print(df['agg_cluster'].value_counts().sort_index())

## 7. Cluster Size Distribution

In [ ]:
cluster_sizes = df['agg_cluster'].value_counts().sort_index()

plt.figure(figsize=(10, 5))
plt.hist(cluster_sizes, bins=30, color='seagreen', edgecolor='black')
plt.xlabel('Cluster size'); plt.ylabel('Number of clusters')
plt.title(f'Agglomerative cluster size distribution (k={BEST_K}, {BEST_LINKAGE} linkage)')
plt.tight_layout()
plt.savefig("../outputs/05_agg_size_histogram.png", dpi=150, bbox_inches="tight")
plt.show()

**Read this against the central question from the top of the notebook:** are these reasonably even, playlist-sized-ish groups (a genuine middle ground DBSCAN never found), or does one cluster dominate the way the DBSCAN mega-cluster did?


## 8. Visualise with t-SNE

In [ ]:
tsne = TSNE(n_components=2, random_state=RANDOM_STATE)
tsne_results = tsne.fit_transform(scaled)
tsne_results['Cluster'] = df['agg_cluster']

sns.relplot(
    data=tsne_results,
    x=tsne_results.columns[0], y=tsne_results.columns[1],
    hue='Cluster', palette='tab10', s=25
).set(title=f"Agglomerative Clustering (k={BEST_K}, {BEST_LINKAGE}) — t-SNE")
plt.savefig("../outputs/05_agg_tsne.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Compare Against K-Means and DBSCAN

In [ ]:
kmeans_baseline = KMeans(n_clusters=8, random_state=RANDOM_STATE, n_init=10)
kmeans_labels = kmeans_baseline.fit_predict(scaled)
kmeans_silhouette = silhouette_score(scaled, kmeans_labels)

# DBSCAN comparison numbers from notebook 04's canonical run (eps=0.20, min_samples=7) — hardcoded here
# for the summary table; re-run 04 if these need refreshing
dbscan_clusters, dbscan_noise, dbscan_silhouette = 2, 417, 0.456

comparison = pd.DataFrame({
    "Method": ["K-Means (k=8)", "DBSCAN (eps=0.20)", f"Agglomerative (k={BEST_K}, {BEST_LINKAGE})"],
    "Clusters": [8, dbscan_clusters, BEST_K],
    "Coverage_%": [100.0, round(100*(1 - dbscan_noise/len(df)), 1), 100.0],
    "Silhouette": [round(kmeans_silhouette, 3), dbscan_silhouette, round(agg_silhouette, 3)]
})
comparison

**Same honesty rule as DBSCAN's comparison:** a higher silhouette alone doesn't mean "better" — check what it's actually describing (many balanced clusters, or one dominant mass distorting the average) before drawing a conclusion.


## 10. Verdict

*(Fill in after running the notebook — does Agglomerative find a workable middle ground DBSCAN missed, or confirm the same "no natural playlist count exists in this data" conclusion a third way? Check Step 7's size distribution directly, not just the silhouette table in Step 9, before deciding.)*

---
**Next:** fold this verdict into the final report and presentation.
